# Cyclone Mocha — IBTrACS Track

Plots the IBTrACS best-track positions for Cyclone Mocha (NOAA, NI basin)
with a time slider. Points accumulate as the slider advances, building
the full track by the last timestep.

**Period:** 2023-05-13 12:00 UTC → 2023-05-14 12:00 UTC

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyproj
import hvplot.pandas
import geoviews.tile_sources as gvts
import holoviews as hv
import panel as pn
from holoviews import opts

hv.extension('bokeh')
pn.extension()

print('Setup complete')

## 2. Load IBTrACS best-track for Cyclone Mocha

In [ ]:
IBTRACS_URL = (
    'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs'
    '/v04r01/access/csv/ibtracs.NI.list.v04r01.csv'
)

T_START = pd.Timestamp('2023-05-13 12:00')
T_END   = pd.Timestamp('2023-05-14 12:00')

ibt = pd.read_csv(IBTRACS_URL, skiprows=[1], low_memory=False)

mocha = (
    ibt[
        (ibt['NAME'].str.upper() == 'MOCHA') &
        (ibt['SEASON'].astype(str) == '2023')
    ]
    .copy()
)

mocha['time'] = pd.to_datetime(mocha['ISO_TIME'])
mocha['lat']  = pd.to_numeric(mocha['LAT'],      errors='coerce')
mocha['lon']  = pd.to_numeric(mocha['LON'],      errors='coerce')
mocha['pres'] = pd.to_numeric(mocha['WMO_PRES'], errors='coerce')
mocha['wind'] = pd.to_numeric(mocha['WMO_WIND'], errors='coerce')
mocha = mocha.dropna(subset=['lat', 'lon']).reset_index(drop=True)

mocha = mocha[(mocha['time'] >= T_START) & (mocha['time'] <= T_END)].reset_index(drop=True)

print(f'Track points: {len(mocha)}')
print(f'Period: {mocha["time"].iloc[0]}  →  {mocha["time"].iloc[-1]}')
mocha[['time', 'lat', 'lon', 'pres', 'wind']]

## 3. Interactive plot — cumulative IBTrACS track

In [ ]:
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x_min, y_min = transformer.transform(90.9, 17.5)
x_max, y_max = transformer.transform(94.6, 21.3)

AREA  = [21.3, 90.9, 17.5, 94.6]   # [N, W, S, E]

times = mocha['time'].values

time_slider = pn.widgets.DiscreteSlider(
    name='Time (UTC)',
    options={str(pd.Timestamp(t)): t for t in times},
    width=700,
)

@pn.depends(time_slider)
def track_plot(t):
    t_ts = pd.Timestamp(t)
    track_so_far = mocha[mocha['time'] <= t_ts]

    points = track_so_far.hvplot.points(
        x='lon', y='lat',
        geo=True,
        c='wind',
        cmap='viridis',
        clim=(20, 120),
        size=60,
        colorbar=True,
        clabel='Wind speed (kt, IBTrACS)',
        hover_cols=['time', 'pres', 'wind'],
        tiles='EsriImagery',
    )

    return points.opts(
        opts.Points(tools=['hover']),
        opts.Overlay(
            width=750, height=600,
            xlim=(x_min, x_max),
            ylim=(y_min, y_max),
            title=f'Cyclone Mocha — {t_ts.strftime("%Y-%m-%d %H:%M UTC")}',
        ),
    )

pn.Column(time_slider, track_plot)